In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

output_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed"

In [4]:
def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df
def load_lane_detection(lane_path):

    lane_df = pd.read_csv(
        lane_path,
        sep=r"\s+",
        header=None
    )

    lane_df.columns = [

        "time",

        "lane_offset",

        "phi",

        "road_width",

        "lane_state"
    ]

    return lane_df

def clean_lane_detection(lane_df):

    lane_df = lane_df.copy()

    lane_df.replace(-9, np.nan, inplace=True)

    lane_df.replace(-99, np.nan, inplace=True)

    lane_df = lane_df.sort_values(
        "time"
    )

    lane_df = lane_df.reset_index(
        drop=True
    )

    return lane_df
def load_vehicle_detection(vehicle_path):

    vehicle_df = pd.read_csv(

        vehicle_path,

        sep=r"\s+",

        header=None

    )

    vehicle_df.columns = [

        "time",

        "front_distance",

        "relative_speed",

        "vehicle_state",

        "confidence"

    ]

    return vehicle_df

In [5]:
def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

def merge_lane_data(master_df, lane_df):

    master_df = master_df.copy()
    lane_df = lane_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    lane_df = lane_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        lane_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    merged_df.drop(
        columns=["time"],
        inplace=True
    )

    return merged_df

def merge_vehicle_data(master_df, vehicle_df):

    master_df = master_df.copy()
    vehicle_df = vehicle_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    vehicle_df = vehicle_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        vehicle_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    merged_df.drop(
        columns=["time"],
        inplace=True
    )

    return merged_df

def clean_vehicle_detection(vehicle_df):

    vehicle_df = vehicle_df.copy()

    vehicle_df = vehicle_df.sort_values(

        "time"

    )

    vehicle_df = vehicle_df.reset_index(

        drop=True

    )

    return vehicle_df

In [6]:
def extract_window_features_v5(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        values = window[feature]

        stats = extract_statistics_v4(
            values,
            feature
        )

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

        # Yeni özellik
        window_stats[f"{feature}_change"] = (
            values.iloc[-1] - values.iloc[0]
        )

    return window_stats

In [9]:
def engineer_features_v6(master_df):

    master_df = master_df.copy()

    # ----------------------------------------------------
    # Acceleration Features
    # ----------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # ----------------------------------------------------
    # Delta Features
    # ----------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    # ----------------------------------------------------
    # Lane Features
    # ----------------------------------------------------

    master_df["lane_departure"] = (
        master_df["lane_offset"].abs() > 0.75
    ).astype(int)

    master_df["steering_alignment"] = (
        master_df["lane_offset"] *
        master_df["phi"]
    )
   # Dynamic Lane Features

    master_df["lane_offset_velocity"] = (
        master_df["lane_offset"]
        .diff()
        .fillna(0)
    )
    # ------------------------------------------------
    # Vehicle Features
    # ------------------------------------------------

    master_df["vehicle_present"] = (
        master_df["front_distance"] > 0
    ).astype(int)

    master_df["safe_distance"] = np.where(
        master_df["vehicle_present"] == 1,
        master_df["front_distance"],
        np.nan
    )

    master_df["closing_speed"] = np.where(
        master_df["vehicle_present"] == 1,
        master_df["relative_speed"],
        0
    )

    master_df["ttc"] = np.where(
        (master_df["vehicle_present"] == 1) &
        (master_df["closing_speed"] > 0),
        master_df["front_distance"] /
        master_df["closing_speed"],
        np.nan
    )
    # ---------------------------------------------
    # Dynamic Vehicle Features
    # ---------------------------------------------

    master_df["ttc_trend"] = (
        master_df["ttc"]
        .diff()
        .fillna(0)
   )

    return master_df


In [10]:
def extract_statistics_v4(signal, feature_name):

    features = {}

    binary_features = [

        "vehicle_present",

        "lane_departure"

    ]

    if feature_name in binary_features:

        features["mean"] = signal.mean()

        return features

    # ----------------------------------------------------
    # Continuous Features
    # ----------------------------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

In [12]:
def create_sliding_windows_v5(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v5(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )

In [19]:
def extract_window_features_v5(window, feature_list):

    window_stats = {}

    change_features = {
        "speed",
        "heading",
        "yaw",
        "lane_offset",
        "ttc"
    }

    for feature in feature_list:

        values = window[feature]

        stats = extract_statistics_v4(
            values,
            feature
        )

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

        # ----------------------------------------
        # Start-End Delta Features
        # ----------------------------------------

        if feature in change_features:

            window_stats[f"{feature}_change"] = (
                values.iloc[-1] - values.iloc[0]
            )

    return window_stats

In [14]:
def process_trip_v7(
    dataset_path,
    driver,
    trip,
    window_size=60
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    # -----------------------------
    # File Paths
    # -----------------------------

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    lane_path = os.path.join(
        trip_path,
        "PROC_LANE_DETECTION.txt"
    )

    vehicle_path = os.path.join(
        trip_path,
        "PROC_VEHICLE_DETECTION.txt"
    )

    # -----------------------------
    # Load
    # -----------------------------

    acc_df = load_accelerometer(acc_path)

    gps_df = load_gps(gps_path)

    lane_df = clean_lane_detection(
        load_lane_detection(lane_path)
    )

    vehicle_df = clean_vehicle_detection(
        load_vehicle_detection(vehicle_path)
    )

    # -----------------------------
    # Merge Sensors
    # -----------------------------

    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    master_df = merge_lane_data(
        master_df,
        lane_df
    )

    master_df = merge_vehicle_data(
        master_df,
        vehicle_df
    )

    # -----------------------------
    # Feature Engineering
    # -----------------------------

    feature_df = engineer_features_v6(
        master_df
    )

    # -----------------------------
    # Sliding Windows
    # -----------------------------

    window_dataset = create_sliding_windows_v5(
        feature_df,
        window_features_v7,
        window_size
    )

    # -----------------------------
    # Metadata
    # -----------------------------

    window_dataset["driver"] = driver

    window_dataset["trip"] = trip

    road_type = trip.split("-")[-1]

    behavior = trip.split("-")[-2]

    window_dataset["road_type"] = road_type

    window_dataset["behavior"] = behavior

    return window_dataset

In [17]:
window_features_v7 = [

    # Acceleration
    "acc_resultant",
    "acc_horizontal",

    # GPS
    "speed",
    "speed_delta",

    "roll",
    "pitch",
    "yaw",

    # Lane
    "lane_offset",
    "phi",
    "steering_alignment",
    "lane_offset_velocity",   # <-- yeni

    # Vehicle
    "front_distance",
    "relative_speed",
    "closing_speed",
    "ttc",
    "ttc_trend",              # <-- yeni

    # Binary Features
    "vehicle_present",
    "lane_departure"
]

print(window_features_v7)
print(len(window_features_v7))

['acc_resultant', 'acc_horizontal', 'speed', 'speed_delta', 'roll', 'pitch', 'yaw', 'lane_offset', 'phi', 'steering_alignment', 'lane_offset_velocity', 'front_distance', 'relative_speed', 'closing_speed', 'ttc', 'ttc_trend', 'vehicle_present', 'lane_departure']
18


In [20]:
sample_dataset = process_trip_v7(
    dataset_path,
    "D1",
    "20151110175712-16km-D1-NORMAL1-SECONDARY",
    60
)

print(sample_dataset.shape)

sample_dataset.filter(regex="change").head()

(6111, 202)


,speed_change,yaw_change,lane_offset_change,ttc_change
0,-4.0,0.048,0.081,NaN
1,-4.0,0.050,0.040,NaN
2,-4.0,0.053,0.010,NaN
3,-4.0,0.056,-0.010,-0.98668
4,-4.0,0.058,-0.020,-0.99596


In [21]:
print(sample_dataset.filter(regex="change").columns.tolist())

['speed_change', 'yaw_change', 'lane_offset_change', 'ttc_change']


In [22]:
trip_list = []

for driver in sorted(os.listdir(dataset_path)):

    if not driver.startswith("D"):
        continue

    driver_path = os.path.join(dataset_path, driver)

    if not os.path.isdir(driver_path):
        continue

    for trip in sorted(os.listdir(driver_path)):

        trip_path = os.path.join(driver_path, trip)

        if os.path.isdir(trip_path):

            trip_list.append({
                "driver": driver,
                "trip": trip
            })

print(len(trip_list))

40


In [23]:
from sklearn.model_selection import train_test_split

train_trips, test_trips = train_test_split(
    trip_list,
    test_size=0.20,
    random_state=42
)

print(len(train_trips))
print(len(test_trips))

32
8


In [24]:
train_datasets = []

for trip in train_trips:

    print(
        f"Processing Train : {trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v7(
        dataset_path,
        trip["driver"],
        trip["trip"],
        60
    )

    train_datasets.append(trip_dataset)

Processing Train : D6 - 20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
Processing Train : D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing Train : D4 - 20151204152848-25km-D4-NORMAL-MOTORWAY
Processing Train : D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing Train : D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing Train : D5 - 20151211162829-16km-D5-NORMAL1-SECONDARY
Processing Train : D5 - 20151211170502-16km-D5-DROWSY-SECONDARY
Processing Train : D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing Train : D3 - 20151126125458-16km-D3-NORMAL2-SECONDARY
Processing Train : D4 - 20151203175637-17km-D4-DROWSY-SECONDARY
Processing Train : D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing Train : D5 - 20151211165606-12km-D5-AGGRESSIVE-SECONDARY
Processing Train : D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing Train : D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing Train : D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing Train : D5 -

In [25]:
train_dataset = pd.concat(
    train_datasets,
    ignore_index=True
)

print(train_dataset.shape)

train_dataset.head()

(242965, 202)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_trend_kurtosis,ttc_trend_q25,ttc_trend_q75,ttc_trend_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.062950,0.037494,0.001406,0.012961,0.178804,0.058821,0.073110,0.994585,0.711390,0.033116,...,0.0,0.0,0.0,0.0,0.000000,0.283333,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
1,0.060696,0.034344,0.001179,0.012961,0.144686,0.056502,0.069598,0.821860,0.113281,0.033116,...,0.0,0.0,0.0,0.0,0.000000,0.300000,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
2,0.060563,0.034495,0.001190,0.012961,0.144686,0.056502,0.069556,0.807773,0.093973,0.033116,...,0.0,0.0,0.0,0.0,0.000000,0.316667,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
3,0.059966,0.034357,0.001180,0.012961,0.144686,0.053694,0.068969,0.863860,0.203253,0.033116,...,0.0,0.0,0.0,0.0,0.000000,0.316667,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
4,0.059171,0.033793,0.001142,0.012961,0.144686,0.053694,0.068001,0.933059,0.452223,0.033116,...,0.0,0.0,0.0,0.0,0.016667,0.316667,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE


In [26]:
test_datasets = []

for trip in test_trips:

    print(
        f"Processing Test : {trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v7(
        dataset_path,
        trip["driver"],
        trip["trip"],
        60
    )

    test_datasets.append(trip_dataset)

Processing Test : D3 - 20151126132013-17km-D3-DROWSY-SECONDARY
Processing Test : D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing Test : D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing Test : D4 - 20151204154908-25km-D4-AGGRESSIVE-MOTORWAY
Processing Test : D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing Test : D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing Test : D6 - 20151221112434-17km-D6-NORMAL-SECONDARY
Processing Test : D4 - 20151204160823-25km-D4-DROWSY-MOTORWAY


In [27]:
test_dataset = pd.concat(
    test_datasets,
    ignore_index=True
)

print(test_dataset.shape)

test_dataset.head()

(66060, 202)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_trend_kurtosis,ttc_trend_q25,ttc_trend_q75,ttc_trend_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.069472,0.041229,0.001700,0.009487,0.146826,0.056582,0.080609,0.716307,-0.647953,0.040467,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
1,0.067631,0.040168,0.001613,0.009487,0.146826,0.055759,0.078489,0.781847,-0.482125,0.039345,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
2,0.066677,0.040049,0.001604,0.009487,0.146826,0.053646,0.077608,0.849096,-0.373484,0.039345,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
3,0.064657,0.038979,0.001519,0.009487,0.146826,0.051527,0.075330,0.909001,-0.183656,0.037702,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
4,0.062385,0.038071,0.001449,0.009487,0.146826,0.050660,0.072919,0.934266,-0.004782,0.035844,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY


In [28]:
train_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v7_start_end.csv",
    index=False
)

test_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v7_start_end.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!
